In [23]:
### Import Packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
from scipy.stats import norm

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (accuracy_score,
                                classification_report,
                                confusion_matrix,
                                roc_auc_score,
                                roc_curve)

from sklearn.preprocessing import LabelEncoder

from itertools import product
from matplotlib.backends.backend_pdf import PdfPages

import statsmodels.formula.api as smf

import statsmodels.api as sm

import joblib

In [24]:
### Import Data
df = pd.read_csv('ED_Cleaned_Data.csv')

In [25]:
### Fix data type:

datetime_cols = [
    'arrival_time',
    'triage_time',
    'init_assess_time',
    'last_contact_time'
]

category_cols = [
    'client_id',
    'visit_id',
    'postal_code',
    'ED',
    'gender',
    'CTAS',
    'disposition',
    'arrival_mode',
    'ED_complaint',
    'chief_complaint',
    'period_of_day',
    'day_of_week',
    'season',
    'holiday',
    'admitted'
]

float_cols = [
    'age',
    'tot_lab_orders',
    'tot_lab_test_ordered',
    'avg_lab_order_time_min',
    'tot_diag_orders',
    'tot_diag_test_ordered',
    'avg_diag_order_time_min',
    'LOS_Hr',
    'wait_time_Hr',
    'service_time_min',
    'previous_visits'
]

df['service_time'] = pd.to_timedelta(
    df['service_time'],
    errors='coerce'
)

In [26]:
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

for col in category_cols:
    df[col] = df[col].astype('category')

for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911055 entries, 0 to 911054
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype          
---  ------                   --------------   -----          
 0   client_id                911055 non-null  category       
 1   visit_id                 911055 non-null  category       
 2   postal_code              899101 non-null  category       
 3   ED                       911055 non-null  category       
 4   gender                   911055 non-null  category       
 5   age                      911055 non-null  float64        
 6   arrival_time             911055 non-null  datetime64[ns] 
 7   triage_time              910898 non-null  datetime64[ns] 
 8   CTAS                     911055 non-null  category       
 9   tot_lab_orders           469110 non-null  float64        
 10  tot_lab_test_ordered     469110 non-null  float64        
 11  avg_lab_order_time_min   439750 non-null  float64        
 12  to

#### Disposition

In [28]:
# Keep original data intact
df_dis = df.copy()

In [29]:
# Features available at triage
admit_features = [
    'CTAS',
    'age',
    'arrival_mode',
    'ED',
    'previous_visits',
    'ED_complaint'
]

# Create a separate modeling dataframe
admit_model_df = df_dis[admit_features + ['admitted']].dropna().copy()

# Convert target to numeric
admit_model_df['admitted'] = admit_model_df['admitted'].astype(int)

# One-hot encode predictors
X_admit = pd.get_dummies(
    admit_model_df[admit_features],
    drop_first=True
)

y_admit = admit_model_df['admitted']

In [30]:
# Fit logistic regression on complete modeling rows
admit_model = LogisticRegression(
    max_iter=3000,
    solver='saga',
    n_jobs=-1,
    random_state=42
)

In [31]:
admit_model.fit(X_admit, y_admit)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'saga'
,max_iter,3000
,multi_class,'deprecated'


In [32]:
# Predict admission probability
admit_prob = admit_model.predict_proba(X_admit)[:, 1]

# Predict admission class using cutoff = 0.4
admit_cutoff = 0.4

admit_pred = np.where(
    admit_prob >= admit_cutoff,
    'admitted',
    'not admitted'
)

In [33]:
# Add empty columns to the original dataframe
df_dis['admit_prob'] = np.nan
df_dis['admit_pred'] = pd.Series(index=df_dis.index, dtype='object')

# Add predictions back using the same row index
df_dis.loc[admit_model_df.index, 'admit_prob'] = admit_prob
df_dis.loc[admit_model_df.index, 'admit_pred'] = admit_pred

# Convert to category
df_dis['admit_pred'] = df_dis['admit_pred'].astype('category')

In [34]:
# Check
df_dis[['admitted', 'admit_prob', 'admit_pred']].head()

,admitted,admit_prob,admit_pred
0,1,0.584960,admitted
1,1,0.152501,not admitted
2,1,0.271762,not admitted
3,0,0.055099,not admitted
4,1,0.283439,not admitted


#### Workload Intensity

In [47]:
# Make a copy
df_sim = df_dis.copy()

In [48]:
### Step 1: Estimate workload weights from observed data

impact_df = df_sim[['service_time_min', 'tot_lab_orders', 'tot_diag_orders']].copy()

impact_df['tot_lab_orders'] = impact_df['tot_lab_orders'].fillna(0)
impact_df['tot_diag_orders'] = impact_df['tot_diag_orders'].fillna(0)

impact_df = impact_df.dropna(subset=['service_time_min'])
impact_df = impact_df[impact_df['service_time_min'] > 0]

impact_df['log_service_time'] = np.log1p(impact_df['service_time_min'])

X_impact = impact_df[['tot_lab_orders', 'tot_diag_orders']]
X_impact = sm.add_constant(X_impact)

y_impact = impact_df['log_service_time']

impact_model = sm.OLS(y_impact, X_impact).fit()

lab_weight = impact_model.params['tot_lab_orders']
diag_weight = impact_model.params['tot_diag_orders']

print("Lab weight:", lab_weight)
print("Diagnostic weight:", diag_weight)

Lab weight: 0.20606980965173172
Diagnostic weight: 0.44093220348123935


In [49]:
### Step 2: Create an observed workload group using observed orders

df_sim['lab_orders_obs'] = df_sim['tot_lab_orders'].fillna(0)
df_sim['diag_orders_obs'] = df_sim['tot_diag_orders'].fillna(0)

df_sim['workload_score_obs'] = (
    lab_weight * df_sim['lab_orders_obs'] +
    diag_weight * df_sim['diag_orders_obs']
)

# Tertile cutoffs
q33 = df_sim['workload_score_obs'].quantile(1/3)
q66 = df_sim['workload_score_obs'].quantile(2/3)

print("Low/Medium cutoff:", q33)
print("Medium/High cutoff:", q66)

def assign_tertile_group(score):
    if score <= q33:
        return 'Low'
    elif score <= q66:
        return 'Medium'
    else:
        return 'High'

df_sim['workload_group_obs'] = df_sim['workload_score_obs'].apply(assign_tertile_group)

Low/Medium cutoff: 0.0
Medium/High cutoff: 0.8818644069624787


In [50]:
### Step 3: Predict lab and diagnostic orders using triage variables

workload_features = [
    'gender',
    'age',
    'CTAS',
    'arrival_mode',
    'ED_complaint',
    'previous_visits'
]

workload_model_df = df_sim[
    workload_features + ['lab_orders_obs', 'diag_orders_obs']
].dropna().copy()

X_workload = pd.get_dummies(
    workload_model_df[workload_features],
    drop_first=True
)

y_lab = workload_model_df['lab_orders_obs']
y_diag = workload_model_df['diag_orders_obs']

lab_model = LinearRegression()
diag_model = LinearRegression()

lab_model.fit(X_workload, y_lab)
diag_model.fit(X_workload, y_diag)

pred_lab = lab_model.predict(X_workload)
pred_diag = diag_model.predict(X_workload)

pred_lab = np.clip(pred_lab, 0, None)
pred_diag = np.clip(pred_diag, 0, None)

In [51]:
### Step 4: Add predicted workload group

df_sim['pred_lab_orders'] = np.nan
df_sim['pred_diag_orders'] = np.nan
df_sim['workload_score_pred'] = np.nan
df_sim['workload_group_pred'] = pd.Series(index=df_sim.index, dtype='object')

df_sim.loc[workload_model_df.index, 'pred_lab_orders'] = pred_lab
df_sim.loc[workload_model_df.index, 'pred_diag_orders'] = pred_diag

df_sim.loc[workload_model_df.index, 'workload_score_pred'] = (
    lab_weight * pred_lab +
    diag_weight * pred_diag
)

# Apply same tertile cutoffs from observed workload score
df_sim.loc[workload_model_df.index, 'workload_group_pred'] = df_sim.loc[
    workload_model_df.index, 'workload_score_pred'
].apply(assign_tertile_group)

df_sim['workload_group_obs'] = df_sim['workload_group_obs'].astype('category')
df_sim['workload_group_pred'] = df_sim['workload_group_pred'].astype('category')

In [52]:
df_sim['workload_group_obs'].value_counts()

workload_group_obs
Low       369387
High      299829
Medium    241839
Name: count, dtype: int64

In [53]:
df_sim['workload_group_pred'].value_counts()

workload_group_pred
Medium    541344
High      360426
Low         9285
Name: count, dtype: int64

In [54]:
df_sim.head(20)

,client_id,visit_id,postal_code,ED,gender,age,arrival_time,triage_time,CTAS,tot_lab_orders,...,admit_prob,admit_pred,lab_orders_obs,diag_orders_obs,workload_score_obs,workload_group_obs,pred_lab_orders,pred_diag_orders,workload_score_pred,workload_group_pred
0,30502,847242,R2K0J4,SBGH,Male,91.0,2011-12-29 15:21:00,2011-12-29 15:23:00,2,5.0,...,0.584960,admitted,5.0,2.0,1.912213,High,5.289730,0.654626,1.378699,High
1,366687,917369,R3J1A5,SBGH,Male,89.0,2012-01-31 11:37:00,2012-01-31 11:41:00,3,7.0,...,0.152501,not admitted,7.0,0.0,1.442489,High,3.965631,1.007836,1.261584,High
2,296828,204268,R2M0H2,SBGH,Male,69.0,2012-02-05 23:16:00,2012-02-05 23:19:00,3,6.0,...,0.271762,not admitted,6.0,2.0,2.118283,High,4.947934,1.082483,1.496921,High
3,339496,164811,R0B0T0,HSC-P,Female,13.0,2012-02-07 18:01:00,2015-11-25 17:46:00,3,NaN,...,0.055099,not admitted,0.0,0.0,0.000000,Low,0.943653,0.149958,0.260580,Medium
4,46099,615834,R2J2A9,SBGH,Male,98.0,2012-06-07 17:18:00,2012-06-07 17:21:00,4,9.0,...,0.283439,not admitted,9.0,2.0,2.736493,High,5.295044,1.208537,1.624032,High
5,7970,450585,R2J2L4,SBGH,Male,84.0,2012-07-03 16:19:00,2012-07-03 16:23:00,4,34.0,...,0.193169,not admitted,34.0,2.0,7.888238,High,4.467705,1.119909,1.414463,High
6,340475,531228,R2H2Z2,SBGH,Male,97.0,2012-07-09 22:43:00,2012-07-09 23:13:00,4,7.0,...,0.557699,admitted,7.0,1.0,1.883421,High,5.268590,0.754862,1.418540,High
7,186680,661838,R3G2N2,GH,Male,80.0,2012-08-09 12:19:00,2012-08-09 12:28:00,2,14.0,...,0.428305,admitted,14.0,0.0,2.884977,High,6.018528,1.281779,1.805414,High
8,112661,154583,R2N1C4,SBGH,Male,80.0,2012-08-17 10:25:00,2012-08-17 10:27:00,2,7.0,...,0.515328,admitted,7.0,0.0,1.442489,High,5.016040,0.560896,1.280971,High
9,367461,13189,R3J2X4,GH,Female,80.0,2012-09-24 15:50:00,2012-09-25 16:07:00,3,7.0,...,0.662180,admitted,7.0,1.0,1.883421,High,5.405725,0.959207,1.536902,High


In [55]:
df_sim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 911055 entries, 0 to 911054
Data columns (total 44 columns):
 #   Column                   Non-Null Count   Dtype          
---  ------                   --------------   -----          
 0   client_id                911055 non-null  category       
 1   visit_id                 911055 non-null  category       
 2   postal_code              899101 non-null  category       
 3   ED                       911055 non-null  category       
 4   gender                   911055 non-null  category       
 5   age                      911055 non-null  float64        
 6   arrival_time             911055 non-null  datetime64[ns] 
 7   triage_time              910898 non-null  datetime64[ns] 
 8   CTAS                     911055 non-null  category       
 9   tot_lab_orders           469110 non-null  float64        
 10  tot_lab_test_ordered     469110 non-null  float64        
 11  avg_lab_order_time_min   439750 non-null  float64        
 12  to

In [56]:
### Save the data for simulation

df_sim.to_excel(
    "ED_Simulation_Data.xlsx",
    index=False
)